In [0]:
#Load the invoice text files into Bronze
import os
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "capstone_supply_chain"
pdf_vol = f"/Volumes/{CATALOG}/bronze/landing_pdf"

# 1. Find the .txt files in the box
files = [f for f in os.listdir(pdf_vol) if f.endswith(".txt")]
print("Files found:", files)

# 2. Read each file completely and keep it as one record
records = []
for f in files:
    filepath = os.path.join(pdf_vol, f)
    with open(filepath, "r") as fh:
        content = fh.read()
    records.append({
        "filename": f,
        "file_path": filepath,
        "raw_content": content,
        "content_length": len(content),
        "file_type": "invoice_pdf_text",
    })

# 3. Turn the list into a Spark table and add audit columns
df_invoices = (spark.createDataFrame(records)
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_file_format", lit("pdf_text"))
)

# 4. Save to Bronze
df_invoices.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.raw_invoices")
print(f"✅ Ingested {len(records)} invoice files → {CATALOG}.bronze.raw_invoices")
display(df_invoices.select("filename", "content_length", "file_type", "_ingested_at"))

In [0]:
#run this to look at the raw text:
print(df_invoices.first()["raw_content"])